# QLoRA fine-tune of Qwen2.5-3B-Instruct (run on Google Colab)

This notebook trains a small **LoRA adapter** on a **free Colab T4 GPU**, then downloads it so you can run
it locally in the main `local_llm_lab.ipynb`.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.

We use [Unsloth](https://github.com/unslothai/unsloth), which makes QLoRA fast and memory-light enough for
the free tier.

## 1. Install dependencies (Colab)

In [ ]:
# Takes a couple of minutes on first run.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes

## 2. Confirm the GPU

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set Runtime to T4 GPU!")

## 3. Load the base model in 4-bit (QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length  = max_seq_length,
    dtype           = None,    # auto
    load_in_4bit    = True,    # QLoRA
)

# Attach LoRA adapters (these are the only weights we train)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. Upload your dataset

Run this and pick the `train.jsonl` you edited in the main project
(`finetune/data/train.jsonl`).

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose train.jsonl
fname = list(uploaded.keys())[0]
print("Uploaded:", fname)

## 5. Build the training set

We apply Qwen's chat template to each `{"messages": [...]}` record to get a single training string.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=fname, split="train")

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds = ds.map(format_chat)
print("examples:", len(ds))
print(ds[0]["text"][:400])

## 6. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,            # small dataset -> a few dozen steps is plenty
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer.train()

## 7. Quick sanity check

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{"role":"user","content":"What is the capital of France?"}]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                       return_tensors="pt", return_dict=True).to("cuda")
out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print(tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 8. Save the adapter and download it

This saves just the small LoRA adapter (a few MB), zips it, and downloads it. Unzip it into
`models/adapters/qwen2.5-3b-tldr/` in your local project.

In [ ]:
model.save_pretrained("qwen2.5-3b-tldr")       # adapter only
tokenizer.save_pretrained("qwen2.5-3b-tldr")

import shutil
shutil.make_archive("qwen2.5-3b-tldr", "zip", "qwen2.5-3b-tldr")

from google.colab import files
files.download("qwen2.5-3b-tldr.zip")

## Optional: export to GGUF for Ollama

If you'd rather run the fine-tuned model through Ollama (fast path), Unsloth can merge + export to GGUF:

```python
model.save_pretrained_gguf("qwen2.5-3b-tldr-gguf", tokenizer, quantization_method="q4_k_m")
```

Then create an Ollama `Modelfile` pointing at the `.gguf` and `ollama create`. See Unsloth's docs.